In [1]:
from typing import TypedDict, List, Annotated
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, AIMessage

from langgraph.graph.message import add_messages

from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

In [4]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")


In [5]:
def process(state: AgentState) -> dict:
    """this node solves the request from input

    Args:
        state (AgentState): _description_

    Returns:
        dict: _description_
    """
    response = llm.invoke(state["messages"])
    #add messages acts as a reducer function, no need to manage manually
    return {"messages": [response]}


In [16]:
from langgraph.checkpoint.memory import MemorySaver


graph = StateGraph(AgentState)

graph.add_node('process', process)
graph.add_edge(START, 'process')
graph.add_edge('process', END)

memory = MemorySaver()

agent = graph.compile(checkpointer=memory)

In [15]:
#this prints everything at once

from langchain_core.runnables import RunnableConfig

config: RunnableConfig  = {"configurable": {"thread_id": "random_id_actual_id_recommended_cant_be_asked_123"}}

user_input = input('Enter: ')
while user_input != 'q':
    print(user_input)
    response = agent.invoke(
        {"messages": [HumanMessage(content=user_input)]},
        config=config  # langgraph fetches and appends history automatically
    )
    response = response['messages'][-1].content
    print(response)
    # print(ai)
    user_input = input('Enter: ')

hello 
Hello there! How can I help you today? 😊
im testing out memory stuff
That's great! I'm happy to help you test out my memory capabilities.

To get started, you can try a few things:

*   **Ask me questions about things we've discussed previously in this conversation.**
*   **Give me some information and then ask me to recall it later.**
*   **Ask me to remember specific details or facts.**

Just let me know what you have in mind, and I'll do my best to remember and respond accurately. What would you like to test first?
my name is miracle
Okay, Miracle! I'll remember that.

What else would you like to test?
what is my name
Your name is Miracle.
